# Smart PPE Monitoring - YOLO vs RT-DETR on Kaggle (T4)

**Before running:** Settings -> Accelerator -> *GPU T4 x1*, Internet -> *On*.
Dataset: attach a YOLOv8-format Hard Hat Workers dataset (the notebook finds `data.yaml` under `/kaggle/input`;
tested listing: `hebaomer/hard-hat-workers`, Roboflow v2, train+test only). Fallback: Kaggle Secret `ROBOFLOW_API_KEY` + `RF_VERSION`.

The project source is **embedded in this notebook** - nothing else to upload. Everything (validation, loading,
training, evaluation, benchmarking) runs here on Kaggle. Set `SMOKE = True` in the setup cell for a quick dry run.

In [ ]:
import sys, subprocess, torch
print(sys.version)
print("torch", torch.__version__, "cuda", torch.version.cuda, "available", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory/2**30, 1), "GiB")
subprocess.run(["nvidia-smi"])

In [ ]:
import os, io, base64, zipfile, shutil
WORK = "/kaggle/working/smart-ppe-monitoring"
BUNDLE = "..."
os.makedirs(WORK, exist_ok=True)
zipfile.ZipFile(io.BytesIO(base64.b64decode(BUNDLE))).extractall(WORK)   # project source is embedded in this notebook
os.chdir(WORK)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)   # torch is preinstalled
SMOKE = False   # True = 1 epoch on 10% of train: tests the whole pipeline and measures time/epoch
EPOCHS = None   # None = config default (50); set an int to give BOTH models the same shorter budget
EXTRA = ("--set train.epochs=1 --set train.fraction=0.1" if SMOKE else
         f"--set train.epochs={EPOCHS}" if EPOCHS else "")


def run(cmd):
    # stops the notebook if the command fails (a plain !cmd would silently continue)
    print("$", cmd, flush=True)
    subprocess.run(cmd, shell=True, check=True)

In [ ]:
# --- dataset: prefer an attached Kaggle dataset (e.g. hebaomer/hard-hat-workers), else Roboflow API ---
import glob
RF_VERSION = None       # fallback only: Roboflow version number, needs Kaggle Secret ROBOFLOW_API_KEY
hits = glob.glob("/kaggle/input/**/data.yaml", recursive=True)
if hits:
    SRC = os.path.dirname(hits[0])
else:
    from kaggle_secrets import UserSecretsClient
    key = UserSecretsClient().get_secret("ROBOFLOW_API_KEY")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "roboflow"], check=True)
    from roboflow import Roboflow
    if RF_VERSION is None:
        raise RuntimeError("No dataset attached under /kaggle/input; attach one or set RF_VERSION")
    SRC = Roboflow(api_key=key).workspace("joseph-nelson").project("hard-hat-workers").version(RF_VERSION)        .download("yolov8", location="/kaggle/working/hardhat_raw").location
print("source dataset:", SRC, os.listdir(SRC))

The Roboflow v2 export has only train+test. This carves a seeded 15% validation split out of *train*; test is copied untouched.

In [ ]:
DST = "/tmp/hardhat"     # outside /kaggle/working so the Output panel only holds results
run(f"python -m smart_ppe.data.make_val_split --src {SRC} --dst {DST}")
os.environ["PPE_DATASET_ROOT"] = DST

In [ ]:
run("python -m smart_ppe.data.validate_dataset --config configs/dataset.yaml")

## Train sequentially (one model on the GPU at a time)
Smoke test first: set `SMOKE = True` in the setup cell.

In [ ]:
run(f"python -m smart_ppe.training.train_yolo --config configs/yolo.yaml {EXTRA}")

In [ ]:
run(f"python -m smart_ppe.training.train_rtdetr --config configs/rtdetr.yaml {EXTRA}")

## Evaluate
Validation split = model selection. **Run the test split once, after choosing configs.**

In [ ]:
for m in ("yolo", "rtdetr"):
    run(f"python -m smart_ppe.evaluation.evaluate --model {m} --weights outputs/runs/{m}/weights/best.pt --split val --name {m}_val_run")

In [ ]:
for m in ("yolo", "rtdetr"):
    run(f"python -m smart_ppe.evaluation.evaluate --model {m} --weights outputs/runs/{m}/weights/best.pt --split test --name {m}")

In [ ]:
run("python -m smart_ppe.evaluation.compare_models --config configs/inference.yaml")

In [ ]:
from IPython.display import Image, display
for f in ("detection_metrics", "speed", "per_class_map", "yolo_training_curves", "rtdetr_training_curves"):
    p = f"outputs/metrics/{f}.png"
    if os.path.exists(p): display(Image(p))

In [ ]:
shutil.make_archive("/kaggle/working/ppe_results", "zip", "outputs")   # appears in the Output panel
print(os.path.getsize("/kaggle/working/ppe_results.zip") / 2**20, "MiB")